# PUMAS - Nairobi Traffic Analysis
## Predictive Urban Mobility Analytics System


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

%matplotlib inline
sns.set_style('whitegrid')

ModuleNotFoundError: No module named 'matplotlib'

## 1. Load and Explore Data

In [ ]:
import sys
sys.path.append('..')

from src.data.data_pipeline import DataPipeline

pipeline = DataPipeline()

### Traffic Flow Data

In [ ]:
traffic_df = pipeline.traffic_df
print(f"Traffic records: {len(traffic_df)}")
traffic_df.head()

### Weather Data

In [ ]:
weather_df = pipeline.weather_df
print(f"Weather records: {len(weather_df)}")
weather_df.head()

## 2. Traffic Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Traffic by Hour
hourly = traffic_df.groupby('hour')['traffic_index'].mean()
axes[0, 0].plot(hourly.index, hourly.values, marker='o', color='#1E88E5')
axes[0, 0].set_title('Average Traffic Index by Hour')
axes[0, 0].set_xlabel('Hour')
axes[0, 0].set_ylabel('Traffic Index')

# Traffic by Zone
zone_traffic = traffic_df.groupby('zone')['traffic_index'].mean().sort_values(ascending=False)
axes[0, 1].barh(zone_traffic.index, zone_traffic.values, color='#43A047')
axes[0, 1].set_title('Traffic Index by Zone')
axes[0, 1].set_xlabel('Traffic Index')

# Speed Distribution
axes[1, 0].hist(traffic_df['avg_speed_kmh'], bins=20, color='#FF7043', edgecolor='white')
axes[1, 0].set_title('Speed Distribution')
axes[1, 0].set_xlabel('Speed (km/h)')
axes[1, 0].set_ylabel('Frequency')

# Congestion Levels
congestion_counts = traffic_df['congestion_level'].value_counts()
axes[1, 1].pie(congestion_counts.values, labels=congestion_counts.index, 
               autopct='%1.1f%%', colors=['#4CAF50', '#FFC107', '#F44336'])
axes[1, 1].set_title('Congestion Level Distribution')

plt.tight_layout()
plt.show()

## 3. Weather Impact Analysis

In [ ]:
weather_summary = weather_df.groupby('weather_condition').agg({
    'temperature_c': 'mean',
    'humidity_percent': 'mean',
    'rain_mm': 'mean'
}).round(2)

print("Weather Condition Summary:")
weather_summary

## 4. Zone Statistics

In [ ]:
zone_stats = pipeline.get_zone_statistics()
zone_stats.sort_values('trip_count', ascending=False)

## 5. Congestion Hotspots

In [ ]:
hotspots = pipeline.get_congestion_hotspots(top_n=5)
print("Top 5 Congestion Hotspots:")
hotspots

## 6. ML Model Testing

In [ ]:
from src.ml.models import WeatherImpactAnalyzer

weather_analyzer = WeatherImpactAnalyzer()
impact = weather_analyzer.analyze_weather_impact(weather_df, traffic_df)

print("Weather Impact Analysis:")
for key, value in impact.items():
    print(f"  {key}: {value}")

---

## Summary

This notebook demonstrates the data analysis capabilities of PUMAS. The system:

1. **Processes traffic flow data** from multiple Nairobi zones
2. **Analyzes weather impact** on traffic patterns
3. **Identifies congestion hotspots** for targeted interventions
4. **Provides zone statistics** for comparative analysis

For real-time visualization, run the Streamlit dashboard:
```bash
streamlit run src/dashboard/app.py
```